# Data Preprocessing

Downloads CourtListener bulk CSVs, filters to federal appellate courts, extracts/normalizes opinion text, writes JSONL shards, and writes a manifest with checksums.


In [1]:
from __future__ import annotations

import bz2
import csv
import datetime as _dt
import hashlib
import json
import os
import re
import sys
import time
import urllib.error
import urllib.request
import xml.etree.ElementTree as ET
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, IO, List, Optional, Set, Tuple

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)

@dataclass
class Config:
    bulk_dir: Path = Path('data/raw/cl_bulk')
    shard_dir: Path = Path('data/raw/cl_federal_appellate_bulk')
    shard_size: int = int(os.environ.get('CL_SHARD_SIZE', '10000'))

    s3_bucket_url: str = 'https://com-courtlistener-storage.s3-us-west-2.amazonaws.com'
    s3_prefix: str = 'bulk-data/'

    min_text_length: int = 50
    text_source_fields: Tuple[str, ...] = (
        'plain_text',
        'html_with_citations',
        'html',
        'html_lawbox',
        'html_columbia',
    )
    log_interval: int = 50_000
    checkpoint_interval: int = 50_000

    @property
    def manifest_path(self) -> Path:
        return self.shard_dir / 'manifest.json'

cfg = Config()
FEDERAL_APPELLATE_COURT_IDS: Set[str] = {
    'ca1','ca2','ca3','ca4','ca5','ca6','ca7','ca8','ca9','ca10','ca11','cadc','cafc'
}

print('Project root:', PROJECT_ROOT)
print('Bulk dir:', cfg.bulk_dir)
print('Shard dir:', cfg.shard_dir)
print('Shard size:', cfg.shard_size)


Project root: /content
Bulk dir: data/raw/cl_bulk
Shard dir: data/raw/cl_federal_appellate_bulk
Shard size: 10000


In [2]:
S3_NS = {'s3': 'http://s3.amazonaws.com/doc/2006-03-01/'}
BULK_FILE_PATTERN = re.compile(r'^bulk-data/(?P<name>[a-z\-]+)-(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})\.csv(?:\.bz2)?$')

def _http_get(url: str, timeout: int = 30, max_retries: int = 5) -> bytes:
    last: Optional[Exception] = None
    for attempt in range(max_retries):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as resp:
                return resp.read()
        except (urllib.error.URLError, TimeoutError) as exc:
            last = exc
            if attempt < max_retries - 1:
                time.sleep(1.0 * (2**attempt))
            else:
                raise
    raise last  # type: ignore[misc]

def _parse_s3_listing(xml_bytes: bytes) -> List[Dict[str, Any]]:
    root = ET.fromstring(xml_bytes)
    files: List[Dict[str, Any]] = []
    for content in root.findall('s3:Contents', S3_NS):
        key_el = content.find('s3:Key', S3_NS)
        size_el = content.find('s3:Size', S3_NS)
        if key_el is None or key_el.text is None or size_el is None or size_el.text is None:
            continue
        key = key_el.text
        size = int(size_el.text)
        files.append({'key': key, 'size': size, 'size_mb': size / 1e6})
    return files

def _is_truncated(xml_bytes: bytes) -> bool:
    root = ET.fromstring(xml_bytes)
    el = root.find('s3:IsTruncated', S3_NS)
    return el is not None and el.text == 'true'

def _get_continuation_token(xml_bytes: bytes) -> Optional[str]:
    root = ET.fromstring(xml_bytes)
    el = root.find('s3:NextContinuationToken', S3_NS)
    return el.text if el is not None else None

def list_s3_files() -> List[Dict[str, Any]]:
    files: List[Dict[str, Any]] = []
    token: Optional[str] = None
    while True:
        url = f"{cfg.s3_bucket_url}?prefix={cfg.s3_prefix}&delimiter=/"
        if token:
            url += f"&continuation-token={token}"
        xml_bytes = _http_get(url)
        files.extend(_parse_s3_listing(xml_bytes))
        if _is_truncated(xml_bytes):
            token = _get_continuation_token(xml_bytes)
            if not token:
                break
        else:
            break
    return files

def _parse_bulk_key(key: str) -> Optional[Tuple[str, _dt.date]]:
    m = BULK_FILE_PATTERN.match(key)
    if not m:
        return None
    try:
        d = _dt.date(int(m.group('year')), int(m.group('month')), int(m.group('day')))
    except ValueError:
        return None
    return (m.group('name'), d)

def find_latest(files: List[Dict[str, Any]], name_prefix: str) -> Dict[str, Any]:
    candidates: List[Dict[str, Any]] = []
    target = name_prefix.rstrip('-')
    for f in files:
        parsed = _parse_bulk_key(f['key'])
        if not parsed:
            continue
        name, d = parsed
        if name.startswith(target):
            candidates.append({**f, 'parsed_name': name, 'parsed_date': d})
    if not candidates:
        raise RuntimeError(f'Missing bulk files for prefix={name_prefix!r}')
    candidates.sort(key=lambda x: x['parsed_date'], reverse=True)
    best = candidates[0]
    return {
        'key': best['key'],
        'size': best['size'],
        'size_mb': best['size_mb'],
        'date': best['parsed_date'].isoformat(),
        'name': best['parsed_name'],
    }

def discover_latest_bulk_files() -> Dict[str, Dict[str, Any]]:
    needed = {
        'courts': 'courts-',
        'dockets': 'dockets-',
        'clusters': 'opinion-clusters-',
        'opinions': 'opinions-',
    }
    files = list_s3_files()
    latest: Dict[str, Dict[str, Any]] = {}
    for label, prefix in needed.items():
        latest[label] = find_latest(files, prefix)
    return latest

latest = discover_latest_bulk_files()
for label, info in latest.items():
    print(f"{label:<8} {info['key']}")


courts   bulk-data/courts-2026-03-31.csv.bz2
dockets  bulk-data/dockets-2026-03-31.csv.bz2
clusters bulk-data/opinion-clusters-2026-03-31.csv.bz2
opinions bulk-data/opinions-2026-03-31.csv.bz2


In [3]:
def _download(url: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        print('✓ exists, skipping:', dest)
        return

    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as resp:
        total = resp.headers.get('Content-Length')
        total_bytes = int(total) if total else None
        tmp = dest.with_suffix(dest.suffix + '.part')
        downloaded = 0
        started = time.time()
        with open(tmp, 'wb') as f:
            while True:
                chunk = resp.read(1024 * 1024)
                if not chunk:
                    break
                f.write(chunk)
                downloaded += len(chunk)
                if total_bytes:
                    pct = downloaded / total_bytes * 100
                    rate = downloaded / max(time.time() - started, 1e-6)
                    sys.stdout.write(f"\r  {dest.name}: {pct:6.2f}% ({downloaded/1e6:,.1f}MB) {rate/1e6:,.1f}MB/s")
                    sys.stdout.flush()
        if total_bytes:
            print()
    tmp.replace(dest)
    print('✓ downloaded:', dest)

cfg.bulk_dir.mkdir(parents=True, exist_ok=True)
local_paths: Dict[str, Path] = {}
for label, info in latest.items():
    filename = Path(info['key']).name
    dest = cfg.bulk_dir / filename
    _download(f"{cfg.s3_bucket_url}/{info['key']}", dest)
    local_paths[label] = dest

local_paths


  courts-2026-03-31.csv.bz2: 100.00% (0.1MB) 0.2MB/s
✓ downloaded: data/raw/cl_bulk/courts-2026-03-31.csv.bz2
  dockets-2026-03-31.csv.bz2: 100.00% (4,979.3MB) 15.9MB/s
✓ downloaded: data/raw/cl_bulk/dockets-2026-03-31.csv.bz2
  opinion-clusters-2026-03-31.csv.bz2:   7.06% (173.0MB) 15.3MB/s

KeyboardInterrupt: 

In [4]:
def file_checksum(path: Path, buffer_size: int = 8192) -> str:
    sha = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(buffer_size), b''):
            sha.update(chunk)
    return sha.hexdigest()

def validate_manifest_shards(manifest: Dict[str, Any], shard_dir: Path) -> bool:
    checksums = manifest.get('checksum')
    if not isinstance(checksums, dict):
        return False
    for shard_name, expected in checksums.items():
        shard_path = shard_dir / shard_name
        if not shard_path.exists():
            return False
        if file_checksum(shard_path) != expected:
            return False
    return True

def read_manifest(path: Path) -> Dict[str, Any]:
    if path.exists():
        return json.loads(path.read_text())
    return {}

existing_manifest = read_manifest(cfg.manifest_path)
if existing_manifest and validate_manifest_shards(existing_manifest, cfg.shard_dir):
    print(f"✓ Already complete: {existing_manifest.get('num_cases', 0):,} cases, {existing_manifest.get('num_shards', 0)} shards verified")
else:
    print('No valid manifest found; will extract shards in next cells.')


No valid manifest found; will extract shards in next cells.


In [5]:
def open_csv(path: Path) -> IO[str]:
    if str(path).endswith('.bz2'):
        return bz2.open(path, 'rt', encoding='utf-8', errors='replace', newline='')
    return open(path, 'r', encoding='utf-8', errors='replace', newline='')

_HTML_TAG_RE = re.compile(r'<[^>]+>')
_MULTI_SPACE_RE = re.compile(r'[ \t]+')
_MULTI_NEWLINE_RE = re.compile(r'\n{3,}')
_NULL_BYTE_RE = re.compile(r'\x00')
_MOJIBAKE_RE = re.compile(r'[\x80-\x9f]')
_PAGE_STAR_RE = re.compile(r'\n?\*\d+\n?')
_WESTLAW_HEADER_RE = re.compile(r'^(Not Reported in .*?\n|Only the Westlaw citation.*?\n)', re.MULTILINE)

def strip_html_preserve_citations(text: str) -> str:
    text = _HTML_TAG_RE.sub(' ', text)
    text = _MULTI_SPACE_RE.sub(' ', text)
    return text.strip()

def remove_boilerplate(text: str) -> str:
    text = _WESTLAW_HEADER_RE.sub('', text)
    text = _PAGE_STAR_RE.sub(' ', text)
    return text.strip()

def clean_encoding(text: str) -> str:
    text = _NULL_BYTE_RE.sub('', text)
    text = _MOJIBAKE_RE.sub('', text)
    dq = chr(34)
    sq = chr(39)
    text = text.replace('“', dq).replace('”', dq)
    text = text.replace('‘', sq).replace('’', sq)
    text = text.replace('—', '--').replace('–', '-')
    return text

def normalize_text(raw_text: str, text_source: str) -> Tuple[str, List[str]]:
    flags: List[str] = []
    cleaned = clean_encoding(raw_text)
    if cleaned != raw_text:
        flags.append('encoding_cleaned')
    if text_source != 'plain_text':
        cleaned = strip_html_preserve_citations(cleaned)
        flags.append('html_stripped')
    before = cleaned
    cleaned = remove_boilerplate(cleaned)
    if cleaned != before:
        flags.append('boilerplate_removed')
    if _MULTI_NEWLINE_RE.search(cleaned):
        cleaned = _MULTI_NEWLINE_RE.sub('\n\n', cleaned)
        flags.append('newlines_collapsed')
    cleaned = cleaned.strip()
    if cleaned != raw_text.strip():
        flags.append('whitespace_trimmed')
    return cleaned, flags

def select_best_text(row: Dict[str, Any]) -> Tuple[str, str]:
    for field in cfg.text_source_fields:
        candidate = (row.get(field) or '').strip()
        if len(candidate) >= cfg.min_text_length:
            return candidate, field
    return '', ''

print('Text helpers ready')


Text helpers ready
